# DAY 8 - EDA: Exploratory Data Analysis

**EDA = First thing you do when you get data.**

Goal: Understand the data BEFORE building models.

EDA tells you:
- What columns exist and their types
- Distribution of each variable
- Relationships between variables
- Anomalies and outliers
- Business insights

---

## EDA Framework
```
1. Data Overview (shape, types, sample)
2. Missing Value Analysis
3. Univariate Analysis (one column at a time)
4. Bivariate Analysis (two columns together)
5. Multivariate Analysis (multiple columns)
6. Correlation Analysis
7. Business Insights & Summary
```

Dataset: **E-Commerce Sales Data** (realistic)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 20)
print('Libraries loaded!')

In [ ]:
# Generate realistic e-commerce dataset
np.random.seed(42)
n = 1000

categories = ['Electronics', 'Clothing', 'Books', 'Home & Garden', 'Sports', 'Beauty']
regions = ['North', 'South', 'East', 'West']
ship_modes = ['Standard', 'Express', 'Overnight']
months = pd.date_range('2023-01-01', periods=12, freq='MS')

df = pd.DataFrame({
    'order_id': range(1001, 1001 + n),
    'order_date': np.random.choice(pd.date_range('2023-01-01', '2023-12-31'), n),
    'customer_age': np.random.randint(18, 65, n),
    'customer_gender': np.random.choice(['Male', 'Female', 'Other'], n, p=[0.5, 0.45, 0.05]),
    'category': np.random.choice(categories, n, p=[0.25, 0.20, 0.15, 0.15, 0.15, 0.10]),
    'product_price': np.random.exponential(3000, n).astype(int) + 500,
    'quantity': np.random.randint(1, 6, n),
    'discount_pct': np.random.choice([0, 5, 10, 15, 20, 25], n, p=[0.3, 0.15, 0.25, 0.15, 0.10, 0.05]),
    'region': np.random.choice(regions, n, p=[0.3, 0.25, 0.25, 0.20]),
    'shipping_mode': np.random.choice(ship_modes, n, p=[0.5, 0.35, 0.15]),
    'rating': np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.10, 0.20, 0.40, 0.25]),
    'returned': np.random.choice([True, False], n, p=[0.08, 0.92])
})

# Calculated columns
df['sales_amount'] = (df['product_price'] * df['quantity'] * (1 - df['discount_pct']/100)).astype(int)
df['month'] = df['order_date'].dt.month_name()
df['month_num'] = df['order_date'].dt.month
df['day_of_week'] = df['order_date'].dt.day_name()

# Add some missing values
df.loc[np.random.choice(df.index, 30), 'customer_age'] = np.nan
df.loc[np.random.choice(df.index, 20), 'rating'] = np.nan

print('Dataset Created!')
print(f'Shape: {df.shape}')

---
## STEP 1: Data Overview

In [ ]:
print('='*60)
print('STEP 1: DATA OVERVIEW')
print('='*60)

print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print('\nFirst 5 rows:')
print(df.head())
print('\nLast 3 rows:')
print(df.tail(3))

In [ ]:
print('Column Names & Data Types:')
df.info()
print('\nBasic Statistics:')
print(df.describe())

---
## STEP 2: Missing Value Analysis

In [ ]:
print('='*60)
print('STEP 2: MISSING VALUE ANALYSIS')
print('='*60)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_report[missing_report['Missing Count'] > 0])

fig, ax = plt.subplots(figsize=(8, 4))
missing_nonzero = missing[missing > 0]
ax.bar(missing_nonzero.index, missing_nonzero.values, color='tomato', edgecolor='black')
for i, v in enumerate(missing_nonzero):
    ax.text(i, v + 0.2, str(v), ha='center', fontsize=11)
ax.set_title('Missing Values per Column', fontsize=14)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

---
## STEP 3: Univariate Analysis (One Column at a Time)

In [ ]:
print('='*60)
print('STEP 3: UNIVARIATE ANALYSIS')
print('='*60)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Numeric Variable Distributions', fontsize=18, fontweight='bold')

numeric_cols = ['customer_age', 'product_price', 'quantity', 'discount_pct', 'sales_amount', 'rating']
for ax, col in zip(axes.flatten(), numeric_cols):
    data = df[col].dropna()
    ax.hist(data, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='--', label=f'Median: {data.median():.1f}')
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Categorical column analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Categorical Variable Analysis', fontsize=18, fontweight='bold')

cat_cols = ['category', 'region', 'shipping_mode', 'customer_gender', 'rating', 'returned']
for ax, col in zip(axes.flatten(), cat_cols):
    counts = df[col].value_counts()
    ax.bar(counts.index.astype(str), counts.values, color=plt.cm.Set2.colors[:len(counts)])
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## STEP 4: Bivariate Analysis (Two Variables Together)

In [ ]:
print('='*60)
print('STEP 4: BIVARIATE ANALYSIS')
print('='*60)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Bivariate Analysis', fontsize=18, fontweight='bold')

# 1. Sales by Category
category_sales = df.groupby('category')['sales_amount'].mean().sort_values(ascending=False)
axes[0,0].bar(category_sales.index, category_sales.values, color=plt.cm.Set1.colors[:len(category_sales)])
axes[0,0].set_title('Average Sales by Category')
axes[0,0].set_ylabel('Avg Sales (Rs)')
axes[0,0].tick_params(axis='x', rotation=30)

# 2. Sales by Region
region_sales = df.groupby('region')['sales_amount'].sum()
axes[0,1].pie(region_sales, labels=region_sales.index, autopct='%1.1f%%',
              colors=plt.cm.Pastel1.colors[:4], startangle=90)
axes[0,1].set_title('Total Sales by Region')

# 3. Price vs Sales scatter
axes[1,0].scatter(df['product_price'], df['sales_amount'], alpha=0.3,
                  color='steelblue', s=20)
axes[1,0].set_title('Product Price vs Sales Amount')
axes[1,0].set_xlabel('Product Price (Rs)')
axes[1,0].set_ylabel('Sales Amount (Rs)')

# 4. Discount vs Rating
df_agg = df.groupby('discount_pct')['rating'].mean().dropna()
axes[1,1].bar(df_agg.index, df_agg.values, color='goldenrod', edgecolor='black')
axes[1,1].set_title('Avg Rating by Discount %')
axes[1,1].set_xlabel('Discount %')
axes[1,1].set_ylabel('Average Rating')
axes[1,1].set_ylim(0, 5)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots — Category vs Sales
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='category', y='sales_amount', palette='Set2')
plt.title('Sales Distribution by Category', fontsize=15)
plt.xlabel('Category')
plt.ylabel('Sales Amount (Rs)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Time-based analysis
monthly = df.groupby('month_num').agg(
    total_sales=('sales_amount', 'sum'),
    order_count=('order_id', 'count'),
    avg_order_value=('sales_amount', 'mean')
).reset_index()

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly['month_name'] = [month_names[m-1] for m in monthly['month_num']]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(monthly['month_num'], monthly['total_sales'], 'b-o', linewidth=2)
axes[0].fill_between(monthly['month_num'], monthly['total_sales'], alpha=0.1, color='blue')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_names, rotation=30)
axes[0].set_title('Monthly Total Sales', fontsize=14)
axes[0].set_ylabel('Total Sales (Rs)')
axes[0].grid(True, alpha=0.3)

axes[1].bar(monthly['month_num'], monthly['order_count'], color='steelblue', edgecolor='black')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names, rotation=30)
axes[1].set_title('Monthly Order Count', fontsize=14)
axes[1].set_ylabel('Number of Orders')

plt.tight_layout()
plt.show()

---
## STEP 5: Multivariate Analysis

In [ ]:
print('='*60)
print('STEP 5: MULTIVARIATE ANALYSIS')
print('='*60)

# Sales by Category and Region
pivot = df.pivot_table(values='sales_amount', index='category', columns='region', aggfunc='sum')

plt.figure(figsize=(12, 6))
sns.heatmap(pivot, annot=True, fmt=',', cmap='YlOrRd', linewidths=0.5)
plt.title('Total Sales: Category vs Region', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Gender and Category
fig, ax = plt.subplots(figsize=(12, 5))
pivot2 = df.groupby(['category', 'customer_gender'])['sales_amount'].mean().unstack()
pivot2.plot(kind='bar', ax=ax, colormap='Set1', edgecolor='black', width=0.7)
ax.set_title('Average Sales by Category and Gender', fontsize=15)
ax.set_xlabel('Category')
ax.set_ylabel('Avg Sales (Rs)')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Gender')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## STEP 6: Correlation Analysis

In [ ]:
print('='*60)
print('STEP 6: CORRELATION ANALYSIS')
print('='*60)

numeric_df = df[['customer_age', 'product_price', 'quantity', 'discount_pct', 'sales_amount', 'rating']]
corr = numeric_df.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.show()

print('\nKey Observations:')
print('- sales_amount vs product_price correlation:', round(corr.loc['sales_amount', 'product_price'], 2))
print('- sales_amount vs quantity correlation:', round(corr.loc['sales_amount', 'quantity'], 2))
print('- rating vs discount_pct correlation:', round(corr.loc['rating', 'discount_pct'], 2))

---
## STEP 7: Business Insights (KPI Summary)

In [ ]:
print('='*60)
print('STEP 7: BUSINESS INSIGHTS')
print('='*60)

total_revenue  = df['sales_amount'].sum()
avg_order_val  = df['sales_amount'].mean()
total_orders   = len(df)
return_rate    = df['returned'].mean() * 100
avg_rating     = df['rating'].mean()
best_category  = df.groupby('category')['sales_amount'].sum().idxmax()
best_region    = df.groupby('region')['sales_amount'].sum().idxmax()
best_month_num = df.groupby('month_num')['sales_amount'].sum().idxmax()
best_month     = month_names[best_month_num - 1]

print(f'\n--- KEY PERFORMANCE INDICATORS ---')
print(f'Total Revenue      : Rs {total_revenue:>12,}')
print(f'Total Orders       : {total_orders:>12,}')
print(f'Avg Order Value    : Rs {avg_order_val:>11,.0f}')
print(f'Return Rate        : {return_rate:>11.1f}%')
print(f'Avg Customer Rating: {avg_rating:>11.2f}/5')
print(f'Best Category      : {best_category:>12}')
print(f'Best Region        : {best_region:>12}')
print(f'Best Month         : {best_month:>12}')

In [ ]:
# Top 5 customers by value (simulated)
df['customer_id'] = np.random.randint(1, 201, n)
top_customers = df.groupby('customer_id')['sales_amount'].sum().nlargest(5)
print('\nTop 5 Customers by Revenue:')
for rank, (cid, rev) in enumerate(top_customers.items(), 1):
    print(f'  {rank}. Customer {cid}: Rs {rev:,}')

# Category performance table
print('\nCategory Performance Summary:')
cat_perf = df.groupby('category').agg(
    Orders=('order_id', 'count'),
    Total_Revenue=('sales_amount', 'sum'),
    Avg_Order=('sales_amount', 'mean'),
    Avg_Rating=('rating', 'mean'),
    Return_Count=('returned', 'sum')
).round(1)
cat_perf['Return_Rate%'] = (cat_perf['Return_Count'] / cat_perf['Orders'] * 100).round(1)
print(cat_perf.sort_values('Total_Revenue', ascending=False))

In [ ]:
# Final EDA Dashboard
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('E-Commerce EDA Dashboard 2023', fontsize=20, fontweight='bold')

# 1. Monthly revenue
axes[0,0].plot(range(1,13), monthly['total_sales'], 'b-o', lw=2)
axes[0,0].set_xticks(range(1,13))
axes[0,0].set_xticklabels(month_names, rotation=45, fontsize=8)
axes[0,0].set_title('Monthly Revenue')
axes[0,0].grid(True, alpha=0.3)

# 2. Category share
cat_rev = df.groupby('category')['sales_amount'].sum()
axes[0,1].pie(cat_rev, labels=cat_rev.index, autopct='%1.0f%%',
              colors=plt.cm.Set3.colors, startangle=90)
axes[0,1].set_title('Revenue by Category')

# 3. Sales amount distribution
axes[0,2].hist(df['sales_amount'], bins=40, color='steelblue', edgecolor='white')
axes[0,2].axvline(df['sales_amount'].mean(), color='red', linestyle='--', label='Mean')
axes[0,2].set_title('Sales Distribution')
axes[0,2].legend()

# 4. Region comparison
reg = df.groupby('region')['sales_amount'].sum().sort_values(ascending=False)
axes[1,0].bar(reg.index, reg.values, color=plt.cm.Set1.colors[:4])
axes[1,0].set_title('Revenue by Region')
axes[1,0].set_ylabel('Total Revenue (Rs)')

# 5. Rating distribution
rating_counts = df['rating'].value_counts().sort_index()
axes[1,1].bar(rating_counts.index, rating_counts.values,
              color=['#d73027','#fc8d59','#fee08b','#91bfdb','#4575b4'])
axes[1,1].set_title('Customer Ratings')
axes[1,1].set_xlabel('Rating')
axes[1,1].set_ylabel('Count')

# 6. Return rate by category
return_rate_cat = df.groupby('category')['returned'].mean() * 100
axes[1,2].bar(return_rate_cat.index, return_rate_cat.values, color='tomato', edgecolor='black')
axes[1,2].set_title('Return Rate by Category (%)')
axes[1,2].tick_params(axis='x', rotation=30)
axes[1,2].set_ylabel('%')

plt.tight_layout()
plt.show()

---
## EDA Checklist — Every Analyst Must Do This

| Step | Question | Tools |
|------|----------|-------|
| 1. Overview | How big is the data? What columns? | `.shape`, `.info()`, `.head()` |
| 2. Missing | What is missing? How much? | `.isnull().sum()` |
| 3. Univariate | What's the distribution of each column? | `hist()`, `boxplot()` |
| 4. Bivariate | How do two variables relate? | `scatter()`, `groupby()` |
| 5. Multivariate | Any patterns across 3+ variables? | `heatmap()`, `pairplot()` |
| 6. Correlation | Which columns are strongly related? | `.corr()`, `heatmap()` |
| 7. Insights | What does the data tell the business? | Summary stats, KPIs |

---
## Homework

1. Download the Titanic dataset and run full EDA
2. Answer: What factors affected survival?
3. Create a 3x3 dashboard with key insights
4. Write 5 business recommendations from the data
5. Share your findings as if presenting to a manager